In [16]:
import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

In [17]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.special import logit, expit
from scipy.stats import rankdata
from xgboost import XGBClassifier

In [18]:
train = pd.read_csv("assets/train.csv")
test = pd.read_csv("assets/test.csv")

In [19]:
TARGET = "Churn"
ID_COL = "id"

In [20]:
train[TARGET] = train[TARGET].map({"No":0,"Yes":1}).astype(int)

In [21]:
train["TotalCharges"] = pd.to_numeric(train["TotalCharges"], errors="coerce")
test["TotalCharges"] = pd.to_numeric(test["TotalCharges"], errors="coerce")

In [22]:
train["TotalCharges"].fillna(train["TotalCharges"].median(), inplace=True)
test["TotalCharges"].fillna(test["TotalCharges"].median(), inplace=True)


In [23]:
y = train[TARGET]
test_ids = test[ID_COL]

X = train.drop(columns=[TARGET, ID_COL])
X_test = test.drop(columns=[ID_COL])

In [24]:
# Core financial behaviour
X["ChargePerTenure"] = X["TotalCharges"] / (X["tenure"] + 1)
X_test["ChargePerTenure"] = X_test["TotalCharges"] / (X_test["tenure"] + 1)

# Log transforms
for col in ["MonthlyCharges", "TotalCharges"]:
    X[f"log_{col}"] = np.log1p(X[col])
    X_test[f"log_{col}"] = np.log1p(X_test[col])

# Service count (very strong churn signal)
service_cols = [
    "OnlineSecurity","OnlineBackup","DeviceProtection",
    "TechSupport","StreamingTV","StreamingMovies"
]

In [25]:
for col in service_cols:
    if col in X.columns:
        X[col] = (X[col] == "Yes").astype(int)
        X_test[col] = (X_test[col] == "Yes").astype(int)

X["ServiceCount"] = X[service_cols].sum(axis=1)
X_test["ServiceCount"] = X_test[service_cols].sum(axis=1)

In [26]:
# Contract interactions (controlled)
X["Monthly_x_Contract"] = X["MonthlyCharges"] * (X["Contract"] == "Month-to-month").astype(int)
X_test["Monthly_x_Contract"] = X_test["MonthlyCharges"] * (X_test["Contract"] == "Month-to-month").astype(int)

# One-hot encoding
X_enc = pd.get_dummies(X, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)

X_enc, X_test_enc = X_enc.align(X_test_enc, join="left", axis=1, fill_value=0)

X_enc = X_enc.astype("float32")
X_test_enc = X_test_enc.astype("float32")


In [27]:
gc.collect()

2627

In [28]:
stratify_key = train["Churn"].astype(str) + "_" + train["Contract"].astype(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_logits = np.zeros(len(X_enc))
test_logits = np.zeros(len(X_test_enc))

seeds = [42, 77, 99]

In [29]:
for seed in seeds:

    print(f"\n===== Seed {seed} =====")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_enc, stratify_key)):

        X_train, X_val = X_enc.iloc[train_idx], X_enc.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBClassifier(
            n_estimators=50000,
            learning_rate=0.01,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=8,
            reg_alpha=2,
            min_child_weight=4,
            gamma=0.2,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            device="cuda",
            early_stopping_rounds=1200
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        test_pred = model.predict_proba(X_test_enc)[:, 1]

        oof_logits[val_idx] += logit(np.clip(val_pred, 1e-6, 1 - 1e-6)) / len(seeds)
        test_logits += logit(np.clip(test_pred, 1e-6, 1 - 1e-6)) / (len(seeds) * skf.n_splits)

        print(f"Seed {seed} Fold {fold+1} AUC:", roc_auc_score(y_val, val_pred))



===== Seed 42 =====
Seed 42 Fold 1 AUC: 0.9177499172207949
Seed 42 Fold 2 AUC: 0.9163582774320169
Seed 42 Fold 3 AUC: 0.91644387342702
Seed 42 Fold 4 AUC: 0.9179650366983211
Seed 42 Fold 5 AUC: 0.9155566572639315

===== Seed 77 =====
Seed 77 Fold 1 AUC: 0.9177499172207949
Seed 77 Fold 2 AUC: 0.9163582774320169
Seed 77 Fold 3 AUC: 0.91644387342702
Seed 77 Fold 4 AUC: 0.9179650366983211
Seed 77 Fold 5 AUC: 0.9155566572639315

===== Seed 99 =====
Seed 99 Fold 1 AUC: 0.9177499172207949
Seed 99 Fold 2 AUC: 0.9163582774320169
Seed 99 Fold 3 AUC: 0.91644387342702
Seed 99 Fold 4 AUC: 0.9179650366983211
Seed 99 Fold 5 AUC: 0.9155566572639315


In [30]:
oof = expit(oof_logits)
test_preds = expit(test_logits)


In [31]:
temperature = 1.03
oof = expit(logit(np.clip(oof,1e-6,1-1e-6)) * temperature)
test_preds = expit(logit(np.clip(test_preds,1e-6,1-1e-6)) * temperature)
print("\nFINAL CV AUC:", roc_auc_score(y, oof))


FINAL CV AUC: 0.9168143750394732


In [32]:
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": rankdata(test_preds) / len(test_preds)
})

submission.to_csv("submission.csv", index=False)
print("Submission created successfully")

Submission created successfully
